# Prepare Training Data for YOLOv5
This Jupyter notebook should be used to we prepare data for training YOLOv5. The mask annotations (for different instances) are  used to create the training data in the following steps:
- Read the mask annotations and convert them to bounding boxes.
- Crop the large images to smaller sub-images to keep the bounding box sizes large enough for the model to be able to detect them. 
- Convert the format of the annotations to the format expected by YOLOv5. 

In [ ]:
# load required libraries
import os
from PIL import Image
from IPython.display import display
import numpy as np
import shutil
import cv2
import pandas as pd

## YOLOv5 Configurations
The following cell specified the base path for storing the parsed train/test images and annotations, as well as where the config yaml file for YOLOv5 can be found. This file should be created in advance. It should include the following info:
- The paths (relative to the specified base path above) for storing the train/test data (images and annotations)
- The number of classes
- The classnames

In [ ]:
# output (base) path where the images and labels folders 
# should be created for YOLOv4 
OUTPUT_BASE_PATH = '/home/cellareye/Cellanome/YOLOv5'
# the config file (including the path with respect to the base path above) containing
# the classnames and also the location where the training data should be created
OUTPUT_CONFIG_FILENAME = 'data/cells.yaml' 
# YOLO model input size (square)
YOLOV5_INPUT_SIZE = 640

### Parse the yaml config file

In [ ]:
# open the config file
config_file = open(os.path.join(OUTPUT_BASE_PATH, OUTPUT_CONFIG_FILENAME), 'r')

# extract the 'path' specified in the config
yaml_path_param = os.getcwd()
path_found = False
for i, row in enumerate(config_file):
    row_details = row.split(':')
    if row_details[0].strip() == 'path':
        yaml_path_param = row_details[1].strip()
        print('[INFO]: Path specified in the yaml file: ', yaml_path_param)
        path_found=True
        break
        
if not path_found:
    print('[ERROR]: No path parameter was found in the config file! Data will be stored in the current folder!')

# extract the paths for the train and test data specified in the config
for i, row in enumerate(config_file):
    row_details = row.split(':')
    if row_details[0].strip() == 'train':
        TRAIN_IMAGE_FOLDER = os.path.join(yaml_path_param, row_details[1].strip())
    if row_details[0].strip() == 'val':
        TEST_IMAGE_FOLDER = os.path.join(yaml_path_param, row_details[1].strip())
    if row_details[0].strip() == 'names':
        classnames = eval(row_details[1].strip())
        REVERSE_LABEL_MAP = {classnames[i]: i for i in range(len(classnames))}
        LABEL_MAP = {i:classnames[i] for i in range(len(classnames))}
    if row_details[0].strip() == 'nc':
        num_classes = int(row_details[1].strip())

config_file.close()

if num_classes != len(classnames):
    print('[ERROR]: The number of classes in \'nc\' does not match with the list of classes in \'names\'')
else:
    print('[INFO]: Mapping between names and labels: ', REVERSE_LABEL_MAP)
    print('[INFO]: Make sure each class is labeled with the specified index in the annotated mask!')

### Create the output folders

In [ ]:
folders = TRAIN_IMAGE_FOLDER.split('/')
if 'images' in folders:
    
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Train output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
    
    # replace the "images" in the train/test images folder with "labels" to create the path
    # for the annotations
    ind = len(folders) - [f for f in reversed(folders)].index('images') - 1
    folders[ind] = 'labels'
    
    TRAIN_LABEL_FOLDER = '/'.join(folders)
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Train output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
        
else:
    print('[ERROR]: Invalid name for training folder in \'train\': It should include word \'images\'')


folders = TEST_IMAGE_FOLDER.split('/')
if 'images' in folders:
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Test output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
    
    # replace the "images" in the train/test images folder with "labels" to create the path
    # for the annotations
    ind = len(folders) - [f for f in reversed(folders)].index('images') - 1
    folders[ind] = 'labels'
    
    TEST_LABEL_FOLDER = '/'.join(folders)
    for idx in range(len(folders) + 1):
        if not os.path.exists(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx]))):
            print('[WARNING] Test output folder {} does not exists! Creating one'.format('/'.join(folders[:idx])))
            os.mkdir(os.path.join(OUTPUT_BASE_PATH,  '/'.join(folders[:idx])))
else:
    print('[ERROR]: Invalid name for test folder in \'val\': It should include word \'images\'')

print('Train images folder: %s' %TRAIN_IMAGE_FOLDER)
print('Train labels folder: %s' %TRAIN_LABEL_FOLDER)
print('Test images folder: %s' %TEST_IMAGE_FOLDER)
print('Test labels folder: %s' %TEST_LABEL_FOLDER)

## Prepare Data
### Data model - Reading mask annotations
Note that the following class can only parse the masks from CellPose. This parser only works for One class. We need to update this class after getting sample data from the annotators. 

In [ ]:
# a similar class to pytorch dataset to "parse" the masks and extract the bounding boxes
# but in YOLO format, which is (center_x, center_y, w, h) each normalized to the image's width/height
class CellMaskDataset:
    def __init__(self, images_path: str, masks_path: str, 
                 color_depth: int = 13, 
                 normalize:bool = False) -> None:
        self.images_path = images_path
        self.masks_path = masks_path
        # load all images and masks
        # the assumption is the image and its mask annotation use the same name
        self.imgs = list(sorted(os.listdir(images_path)))
        self.masks = list(sorted(os.listdir(masks_path)))
        # the scaling factor for normalizing the channels after Tensor
        # conversion to get values in [0, 1]
        # this is 2 ^ color_depth - 1, where color_depth is the number of bits
        # use to represent the intensities for each channel
        self.channel_scale = 2 ** color_depth - 1
        self.normalize = normalize
        
        if len(self.imgs) != len(self.masks):
            print("[ERROR]: The list of images and masks are not consistent")
            return
        
        for i, img_filename in enumerate(self.imgs):
            # drop the image/mask filename extension 
            # (anything after the last '.' in the filename is considered as extension)
            img_name = ".".join(img_filename.strip().split('.')[:-1])
            mask_name = ".".join(self.masks[i].strip().split('.')[:-1])
            if img_name != mask_name:
                print("[ERROR]: Inconsistent mask file :{} found for image file: {}".format(mask_name, img_name))
     

    def __getitem__(self, idx: int):
        # load images and masks
        img_path = os.path.join(self.images_path, self.imgs[idx])
        mask_path = os.path.join(self.masks_path, self.masks[idx])
        # read the image, do not change the format
        # depending on the set color_depth, the values will be in [0, 2^color_depth - 1]
        # img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
        img = np.array(Image.open(img_path))
        image_height, image_width = img.shape[:2]
        
        # the assumption here is instances are encoded as different levels
        # with 0 being the background
        # each element in mask is an np.uint16 (unsigned 16 bits)
        # we can read such images using PIL.Image
        # convert to a numpy array
        mask = np.array(Image.open(mask_path))
        
        
        # instances are encoded as different gray levels
        # the results are sorted
        obj_ids = np.unique(mask)
        # first id [0] is the background, so remove it
        obj_ids = obj_ids[1:]
        
        # get bounding box coordinates for each mask
        num_objs = len(obj_ids)
        
        # split the color-encoded mask into a set
        # of binary masks
        masks = mask == obj_ids[:, None, None]
        
        boxes = []
        labels = []
        
        for i in range(num_objs):
            
            pos = np.where(masks[i])
            xmin = np.min(pos[1])
            xmax = np.max(pos[1])
            ymin = np.min(pos[0])
            ymax = np.max(pos[0])
            if xmin < xmax and ymin < ymax:
                boxes.append([xmin, ymin, xmax, ymax])
                # add the label, we support only one class for now
                # note that YOLO labels start with 0, and the mask
                # labels start with 1, so we need to subtract 1 here 
                # also, we are saving the classname here
                labels.append(LABEL_MAP[0])
        
        
        # create a pandas DataFrame for ease of procesing
        annotations_df = pd.DataFrame(columns=['xtl', 'ytl', 'xbr', 'ybr', 'label'])
        annotations_df[['xtl', 'ytl', 'xbr', 'ybr']] = boxes
        annotations_df['label'] = labels

        # convert the returned np.unit32 image to float with values between 0, 1
        if self.normalize:
            # if this flag is set, normalize the image such the the minimum intensity 
            # is mapped to zero, and the maximum is mapped to one
             # convert the image to a numpy array
            img = cv2.normalize(img, img, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX).astype(np.uint8)
        else:    
            # the intensity of the images is color_deptj bits, so we need to divide by 2^color_depth - 1
            img = (255 * img / self.channel_scale).astype(np.uint8)
            
        return  {'name': self.imgs[idx], 'image': img, 'annotations': annotations_df}

    def __len__(self):
        return len(self.imgs)

### Crop into smaller sub-images

In [ ]:
def optimize_crop(annotations_df, crop_coords, dx, dy, w, h, objects_of_interest):
    """
    This function finds the "best" center for the crop with corners (x1, y1), 
    (x2, y2) within the specified range +/-dx and +/-dy such that the bouding boxes
    (specified by annotations_df) either lie completely in or completely out of the crop. 

    Args:
        annotations_df (pandas DataFrame): A DataFrame for the bounding boxes with 
            columns 'xtl', 'ytl', 'xbr', 'ybr' and 'label'.
        crop_coords (4-tuple or 4-element list of int): xtl, ytl, xbr, and ybr 
            box coordinates for cropping.
        dx, dy (int): The limit on moving the center of the crop (+/-).
        w, h (int): The original image size. 
        objects_of_interest (list of string): List of objects of interest.
    """
    
    (x1, y1, x2, y2) = crop_coords
    
    # make sure the crop is within the image and corners are interger
    x1, y1, x2, y2 = int(max(0, x1)), int(max(0, y1)), int(min(w, x2)), int(min(h, y2))
    dx = int(dx)
    dy = int(dy)
    w = int(w)
    h = int(h)
    
    # extend the crop by dx and dy in x and y dimensions on both sizes
    xc1 = max(x1 - dx, 0)
    yc1 = max(y1 - dy, 0)
    xc2 = min(x2 + dx, w)
    yc2 = min(y2 + dy, h)
    
    if xc2 <= xc1 or yc2 <= yc1:
        # incorrect input dimensions
        return None
    
    # make a copy of the annotations dataframe
    df = annotations_df.copy()
    
    # transform the bounding boxes
    df[['xtl', 'xbr']] = df[['xtl', 'xbr']] - xc1
    df[['ytl', 'ybr']] = df[['ytl', 'ybr']] - yc1
    
    crop_width = xc2 - xc1
    crop_height = yc2 - yc1
    
    # remove the bounding boxes that are totally outside the extended cropped image
    # keep the ones that have some non-zero overlap
        
    df = df[df.apply(lambda row: True if 
                     max(0, min(row['xbr'], crop_width) - max(row['xtl'], 0)) * \
                     max(0, min(row['ybr'], crop_height) - max(row['ytl'], 0)) > 0 and \
                     row['label'] in objects_of_interest
                     else False, axis = 1)].reset_index(drop=True)
    
    # for each pixel x_c in x dimension with 0 <= x_c < crop_width, calculate the cost of having one of the 
    # crop boundaries (vertical) at x_c as the number of bounding boxes that the vertical line passing 
    # through x_c crosses
    # we assign a weight in [0, 1] to each box depending on where the line through x_c crosses the box: 
    # 0 if the line is within 25% of the length of the box from either edges and 1 otherwise
    # similarly for the y dimension
    cost_in_x = np.zeros(crop_width)
    cost_in_y = np.zeros(crop_height)
    
    for i, row in df.iterrows():
        xtl = int(row['xtl'])
        ytl = int(row['ytl'])
        xbr = int(row['xbr'])
        ybr = int(row['ybr'])
        
        # skip skinny boxes
        if (xbr - xtl) < 4 or (ybr - ytl) < 4:
            continue
        
        # stepwise weight for the box in x dimension, temp[0] is the weight at x = xtl and 
        # temp[xbr - xtl - 1] is the weight at x = xbr - 1
        # temp[0:k] and temp[xbr - xtl - k + 1:xbr - xtl] are 0.25 and the rest are 1
        k = int((xbr - xtl) / 4)
        temp = [0.25] * k + [1] * (xbr - xtl - 2 * k) + [0.25] * k
        
        """
        # triangular weight for the box in x dimension, temp[0] is the weight at x = xtl and 
        # temp[xbr - xtl - 1] is the weight at x = xbr - 1 (both are zero)
        
        # center of the box
        if (xbr + xtl) % 2 == 0:
            mid_x = (xbr + xtl) // 2 - 1 
            temp = [float(i - xtl) / (mid_x - xtl) for i in range(xtl, mid_x + 1)]
            temp += [float(xbr - i - 1) / (xbr - mid_x - 2) for i in range(mid_x + 1, xbr)]
        else:
            mid_x = (xbr + xtl - 1) // 2 
            temp = [float(i - xtl) / (mid_x - xtl) for i in range(xtl, mid_x + 1)]
            temp += [float(xbr - i - 1) / (xbr - mid_x - 1) for i in range(mid_x + 1, xbr)]
        """
        
        
        # pixels of the bounding box in x dimension that overlaps with [0, crop_width]
        for i in range(xtl, xbr): 
            if i >= 0 and i < crop_width:
                cost_in_x[i] += temp[i - xtl] 
        
        # y dimension
        k = int((ybr - ytl) / 4)
        temp = [0.25] * k + [1] * (ybr - ytl - 2 * k) + [0.25] * k
        
        """
        # triangular weight for the box in y dimension, temp[0] is the weight at y = ytl and 
        # temp[ybr - ytl - 1] is the weight at y = ybr - 1 (both are zero)
        
        # center of the box
        if (ybr + ytl) % 2 == 0:
            mid_y = (ybr + ytl) // 2 - 1 
            temp = [float(i - ytl) / (mid_y - ytl) for i in range(ytl, mid_y + 1)]
            temp += [float(ybr - i - 1) / (ybr - mid_y - 2) for i in range(mid_y + 1, ybr)]
        else:
            mid_y = (ybr + ytl - 1) // 2 
            temp = [float(i - ytl) / (mid_y - ytl) for i in range(ytl, midY + 1)]
            temp += [float(ybr - i - 1) / (ybr - mid_y - 1) for i in range(mid_y + 1, ybr)]
        """
        
        # pixels of the bounding box in x dimension that overlaps with [0, cH]
        for i in range(ytl, ybr): 
            if i >= 0 and i < crop_height:
                cost_in_y[i] += temp[i - ytl] 
       
    # limit the search for the start of the box 
    left_delta = min(dx, x1)
    right_delta = min(dx, w - x2)
    start_cost_x = np.zeros(left_delta + right_delta)
    for i in range(left_delta + right_delta):
        start_cost_x[i] = cost_in_x[i] + cost_in_x[x2 - x1 + i]
    
    if len(start_cost_x) > 0:    
        x1_adjusted = np.argmin(start_cost_x) + x1 - left_delta
    else:
        x1_adjusted = x1
    
     # limit the search for the start of the box 
    top_delta = min(dy, y1)
    bottom_delta = min(dy, h - y2)
    start_cost_y = np.zeros(top_delta + bottom_delta)
    for i in range(top_delta + bottom_delta):
        start_cost_y[i] = cost_in_y[i] + cost_in_y[y2 - y1 + i]
    
    if len(start_cost_y) > 0:    
        y1_adjusted = np.argmin(start_cost_y) + y1 - top_delta
    else:
        y1_adjusted = y1
        
    return (x1_adjusted, y1_adjusted, x1_adjusted + x2 - x1, y1_adjusted + y2 - y1)

# crop function
def crop_and_block(sample, crop_coords, objects_of_interest=None, 
                   block_label='block', keep_area_threshold=0.9):
    """
    Crop the image in a sample for a given crop coordinates and black out 
    the partial bounding boxes the lies on the crop boundary.

    Args:
        sample (dictionary): Input data sample to be cropped. The dictionary
            should include "name", "image", and "annotations" keys for passing 
            the image name, the image itself and the bounding boxes pandas DataFrame
            (with columns 'xtl', 'ytl', 'xbr', 'ybr').
        crop_coords (4-tuple or 4-element list of int): xtl, ytl, xbr, and ybr 
            box coordinates for cropping.
        objects_of_interest (list of string): List of objects of interest.
        block_label (string): Label used for partially blocking areas (hiding during training). 
        keep_area_threshold (float): The threshold on the ratio of the area of the 
            bounding boxes that lie inside the cropped image to keep. Bounding boxes 
            with at least keepAreaThreshold of their area inside the cropped image 
            will be kept. Otherwise, all the  bounding boxes crossing the boundaries 
            of the cropped image will be removed. 'block' labels are kept to be 
            blacked out in the train/test images later. 
        
    """
    
    x1, y1, x2, y2 = crop_coords
    # make a copy of the input to make sure it is not modified
    name, image, df = sample['name'], sample['image'].copy(), sample['annotations'].copy()
    
    h, w = image.shape[:2]
    
    xc1 = int(max(x1, 0))
    yc1 = int(max(y1, 0))
    xc2 = int(min(x2, w))
    yc2 = int(min(y2, h))
    
    if xc2 <= xc1 or yc2 <= yc1:
        # incorrect input dimensions
        return None
    
    # transform the bounding boxes
    df[['xtl', 'xbr']] = df[['xtl', 'xbr']] - xc1
    df[['ytl', 'ybr']] = df[['ytl', 'ybr']] - yc1
    
    # sizes of cropped image
    crop_width = xc2 - xc1
    crop_height = yc2 - yc1
    
    # remove the bounding boxes that are totally outside the cropped image
    # keep the ones that have some non-zero overlap
        
    df = df[df.apply(lambda row: True if 
                     max(0, min(row['xbr'], crop_width) - max(row['xtl'], 0)) * \
                     max(0, min(row['ybr'], crop_height) - max(row['ytl'], 0)) > 0\
                     else False, axis = 1)].reset_index(drop=True)
        

    # identify bounding boxes that would lie inside the newly cropped
    # image by more than keep_area_threshold; these boxes together with the
    # boxes labled as 'block' (without any condition on the overlap with 
    # the cropped sub-image) are kept (we keep the 'block' ones to ensure
    # the whole area will be blocked later outside the code)
    
    # also identify bounding boxes that would lie inside the newly cropped
    # image by less than keep_area_threshold and more than 10%; we are going 
    # to black out the area of these bounding boxes in the image to prevent 
    # the model from seeing partial objects
    
    # for any object with overlap less than 10% with the cropped
    # sub-image, we only remove the bounding box. We allow the model to see the 
    # content of the partial object 

    
    if objects_of_interest is None:
        # consider all objects
        idxs_to_keep = df.apply(lambda row: True \
                                if (min(row['xbr'], crop_width) - max(row['xtl'], 0)) * 
                                (min(row['ybr'], crop_height) - max(row['ytl'], 0)) >= keep_area_threshold * 
                                (row['xbr'] - row['xtl']) * (row['ybr'] - row['ytl']) else False, axis=1)
        idxs_to_block = df.apply(lambda row: True \
                                 if ((min(row['xbr'], crop_width) - max(row['xtl'], 0)) * 
                                     (min(row['ybr'], crop_height) - max(row['ytl'], 0)) < keep_area_threshold * 
                                     (row['xbr'] - row['xtl']) * (row['ybr'] - row['ytl']) and 
                                     (min(row['xbr'], crop_width) - max(row['xtl'], 0)) * 
                                     (min(row['ybr'], crop_height) - max(row['ytl'], 0)) >= 0.1 * 
                                     (row['xbr'] - row['xtl']) * (row['ybr'] - row['ytl'])) or
                                 row['label'] == block_label else False, axis=1)
    else:
        # if objects_of_interest is provided, use it to only keep the ones we need to keep 
        idxs_to_keep = df.apply(lambda row: True \
                                if (min(row['xbr'], crop_width) - max(row['xtl'], 0)) * 
                                (min(row['ybr'], crop_height) - max(row['ytl'], 0)) >= keep_area_threshold * 
                                (row['xbr'] - row['xtl']) * (row['ybr'] - row['ytl']) and 
                                row['label'] in objects_of_interest else False, axis=1)
        idxs_to_block = df.apply(lambda row: True \
                                 if ((min(row['xbr'], crop_width) - max(row['xtl'], 0)) * 
                                     (min(row['ybr'], crop_height) - max(row['ytl'], 0)) < keep_area_threshold * 
                                     (row['xbr'] - row['xtl']) * (row['ybr'] - row['ytl']) and 
                                     (min(row['xbr'], crop_width) - max(row['xtl'], 0)) * 
                                     (min(row['ybr'], crop_height) - max(row['ytl'], 0)) >= 0.1 * 
                                     (row['xbr'] - row['xtl']) * (row['ybr'] - row['ytl']) and 
                                     row['label'] in objects_of_interest) or
                                 row['label'] == block_label  else False, axis=1)
        
    # limit the bounding boxes to image coordinates
    df.loc[df['xtl'] < 0, 'xtl'] = 0
    df.loc[df['ytl'] < 0, 'ytl'] = 0
    df.loc[df['xbr'] > crop_width, 'xbr'] = crop_width
    df.loc[df['ybr'] > crop_height, 'ybr'] = crop_height
    
    # now black out the image on the boxes that should be blacked out
    mask = np.ones((crop_height, crop_width), dtype=np.uint8)
    
    # indicate the areas that should be blacked out from the bounding
    # boxes to be blocked
    for _, row in df[idxs_to_block].iterrows():
        mask[int(row['ytl']):int(row['ybr']), int(row['xtl']):int(row['xbr'])]  = 0
    
    # then overwrite them by the annotated objects that we are going to keep
    # we do this to make sure we are not blacking out anything from the objects
    # that are going to be used for training, and only black out objects that we 
    # are not going to use (not annotated anymore)
    for _, row in df[idxs_to_keep].iterrows():
        mask[int(row['ytl']):int(row['ybr']), int(row['xtl']):int(row['xbr'])]  = 1

    df = df[idxs_to_keep].reset_index(drop=True)
    
    if len(image.shape) > 2:
        # 3 channel image
        mask = mask[:, :, np.newaxis]
    
    return  {'name': name, 'image': image[yc1: yc2, xc1: xc2] * mask, 'annotations': df}

## Putting everything together - Saving the parsed annotations and images

In [ ]:
TRAIN_IMAGES_PATH = '/home/cellareye/Cellanome/Images/img/images'
TRAIN_MASKS_PATH = '/home/cellareye/Cellanome/Images/img/masks'

TEST_IMAGES_PATH = '/home/cellareye/Cellanome/Images/img/test/images'
TEST_MASKS_PATH = '/home/cellareye/Cellanome/Images/img/test/masks'

# use our dataset and defined transformations
train_dataset = CellMaskDataset(images_path=TRAIN_IMAGES_PATH, masks_path=TRAIN_MASKS_PATH, 
                                color_depth = 13, normalize=False)

test_dataset = CellMaskDataset(images_path=TEST_IMAGES_PATH, masks_path=TEST_MASKS_PATH, 
                               color_depth = 13, normalize=False)

In [ ]:
train = False

if train:
    set_desc = 'training'
    image_folder = TRAIN_IMAGE_FOLDER
    label_folder = TRAIN_LABEL_FOLDER
    data_set = train_dataset
else:
    set_desc = 'testing'
    image_folder = TEST_IMAGE_FOLDER
    label_folder = TEST_LABEL_FOLDER
    data_set = test_dataset
    
# keep all the labels in the model label map
objects_of_interest = list(REVERSE_LABEL_MAP.keys())

num_images = 0
num_annotations = 0

# the number of overlapping pixels between crops in each dimension
# this is larger than the largest expected cell size (in fact, 3 times 
# more than the expected size of the cells in breadboard images)

overlap_in_x = 100
overlap_in_y = 100

# the step size for the starting point of each crop in x and y dimension
crop_start_step_x = YOLOV5_INPUT_SIZE - overlap_in_x
crop_start_step_y = YOLOV5_INPUT_SIZE - overlap_in_y

# read the image and the annotations, then parse each
for idx in range(len(data_set)):
    
    sample = data_set[idx]
    # image size
    image_height, image_width = sample["image"].shape[:2]
    
    crop_count = 0
    # overlapping crops
    for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
        for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
            # crop coordinates
            xc_tl = x_start
            yc_tl = y_start
            xc_br = x_start + YOLOV5_INPUT_SIZE
            yc_br = y_start + YOLOV5_INPUT_SIZE
            # make sure we always crop the image with the given size
            # if we get to the boundaries, extend the crop
            # size inside the image to always get the same size crop
            # this is not really needed, but help with capturing more
            # annotations toward the low/right parts of the image
            if xc_br > image_width:
                xc_br = image_width 
                xc_tl = xc_br - YOLOV5_INPUT_SIZE
            if yc_br > image_height:
                yc_br = image_height
                yc_tl = yc_br - YOLOV5_INPUT_SIZE
           
            crop_coords = [xc_tl, yc_tl, xc_br, yc_br]
            
            # optimize the crop
            crop_coords = optimize_crop(sample["annotations"], crop_coords, overlap_in_x, overlap_in_y, 
                                        image_width, image_height, objects_of_interest)
            
            # crop and block the image
            cropped_sample =  crop_and_block(sample, crop_coords, objects_of_interest=objects_of_interest)
            
            # save the cropped image and the annotations for this image, for each image name
            # use _ crop_count
            img_name = ".".join(cropped_sample["name"].strip().split('.')[:-1])
            
            # save the image in jpg format
            crp_img_name = img_name + '_crp_' + str(crop_count) + '.jpg'
            cv2.imwrite(os.path.join(OUTPUT_BASE_PATH, image_folder, crp_img_name), cropped_sample["image"])
            num_images += 1
            
            crop_height, crop_width = cropped_sample["image"].shape[:2]
            # annotation txt file
            crp_annot_name = img_name + '_crp_' + str(crop_count) + '.txt'
            # create the YOLOv5 annotations txt file for the cropped image
            annotation_file = open(os.path.join(OUTPUT_BASE_PATH, label_folder, crp_annot_name), 'w+')
            
            for _, row in cropped_sample["annotations"].iterrows():
                # skip labels that are not in the label map 
                # (not really needed as we have already filtered this above)
                if row['label'] not in REVERSE_LABEL_MAP:
                    continue
                num_annotations += 1
                # YOLOv5 annotation format
                center_x = (row['xtl'] + row['xbr']) / (2.0 * crop_width)
                center_y = (row['ytl'] + row['ybr']) / (2.0 * crop_height)
                w = (row['xbr'] - row['xtl']) / float(crop_width)
                h = (row['ybr'] - row['ytl']) / float(crop_height)
                label = REVERSE_LABEL_MAP[row['label']]
                annotation_file.write(' '.join([str(label), str(center_x), str(center_y), str(w), str(h)]) + '\n')
    
            annotation_file.close()
            
            crop_count += 1
   
    
print('Created {} '.format(num_images) + 'images for ' + set_desc)
print('Created {} '.format(num_annotations) + 'objects for ' + set_desc)

In [ ]:
# checks on the results
def check_results (idx, train = True):
    if train:
        image_folder = TRAIN_IMAGE_FOLDER
        label_folder = TRAIN_LABEL_FOLDER
    else:
        image_folder = TEST_IMAGE_FOLDER
        label_folder = TEST_LABEL_FOLDER
    
    dataset_files = os.listdir(os.path.join(OUTPUT_BASE_PATH, image_folder))
    name = dataset_files[idx]
    
    # colors for displaying bounding boxes
    COLORS = [(0, 0, 255), (0, 255, 0), (255, 0, 0),
              (255, 0, 255), (0, 255, 255), (255, 255, 0)]
    # label map
    LABEL_MAP = {value: key for key, value in REVERSE_LABEL_MAP.items()}
    
    image = cv2.imread(os.path.join(OUTPUT_BASE_PATH, image_folder, name))
    H, W = image.shape[:2]
    print('The image sizes are (H, W) = %d, %d' %(H, W))
    # read the annotations file
    with open(os.path.join(OUTPUT_BASE_PATH, label_folder, name[:-4] + '.txt'),'r') as annot_file:
        
        print('Parsing annotation file %s' %(name[:-4] + '.txt'))
        num_annotations = 0
        for line in annot_file:
            num_annotations += 1
            fields = line.strip().split(' ')

            (label, center_x, center_y, w, h) = fields
            xtl = int((float(center_x) - float(w) / 2.0) * W)
            ytl = int((float(center_y) - float(h) / 2.0) * H)
            xbr = int((float(center_x) + float(w) / 2.0) * W)
            ybr = int((float(center_y) + float(h) / 2.0) * H)
            
            # convert the label to an integer from string
            label = int(label)
            text = LABEL_MAP[label]
            classIds = list(REVERSE_LABEL_MAP.values())
            if label not in classIds:
                print('Incorrect label was found %s' %label)
                # use black for incorrect label
                color = (0, 0, 0)
            else:
                color = COLORS[label]
                
            cv2.rectangle(image, (xtl, ytl), (xbr, ybr), color, 1)
            cv2.putText(image, text, (xtl, ytl + 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    return Image.fromarray(image[:, :, (2, 1, 0)])

In [ ]:
check_results (idx=3, train = True)